# Gift Concierge Agent — End-to-End Test

Tests the full pipeline:
- Intent routing (Groq)
- Catalog search (Qdrant RAG + Groq)
- Logistics check (rule-based + Groq narration)
- Preference update (Groq extraction + Supabase)
- Chitchat (Groq)
- **Reflection loop** (Draft → Reflect → Revise)

In [2]:
import sys
sys.path.insert(0, '../src')

In [3]:
from agents.orchestrator import GiftOrchestrator
from memory.st_store import SupabaseSTStore
from memory.profile_store import SupabaseProfileStore
from memory.rag_store import QdrantRAGStore
from memory.embedder import OpenRouterEmbedder

orch = GiftOrchestrator(
    st_store=SupabaseSTStore(),
    profile_store=SupabaseProfileStore(),
    rag_store=QdrantRAGStore(embedder=OpenRouterEmbedder()),
)
print('Orchestrator ready')

2026-05-10 23:16:13.590 | INFO     | infastructure.db.supabase_client:get_supabase_client:46 - Supabase client initialised (https://rcxpatxxadqnxubzalxe.supabase.co)
2026-05-10 23:16:13.629 | INFO     | memory.rag_store:__init__:117 - QdrantRAGStore initialised — collection: 'kapruka_agent'


Orchestrator ready


## Helper — pretty print response

In [4]:
def show(r):
    print(f'Intent     : {r.intent} ({r.confidence:.0%})')
    print(f'Session    : {r.session_id}')
    print(f'Metadata   : {r.metadata}')
    print()
    print('Reply:')
    print(r.reply)

## Test 1 — Search (catalog + RAG)

In [5]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Find a dinosaur toy for my kid',
)
show(r)

2026-05-10 23:16:15.843 | INFO     | infastructure.observability:_get_langfuse:63 - LangFuse client initialised (https://us.cloud.langfuse.com)
Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 23:16:19.597 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 17:46:20 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoun

Intent     : search (99%)
Session    : s001
Metadata   : {'products_retrieved': 8, 'query_enriched': 'Find a dinosaur toy for my kid | preferences: dark chocolate, orchids | avoid: nuts, flowers | budget LKR 5000', 'recipient': 'Wife', 'reflection_triggered': True, 'violations_found': False}

Reply:
I must admit, the original request was for a dinosaur toy for your kid, but the recipient profile provided is for your wife. Considering her likes and dislikes, I've found a few gift options that might suit her, although they may not be exactly what you initially requested.

My top recommendation is the Sweet Chirp Tweety Plush (LKR 2,500, https://www.kapruka.com/buyonline/sweet-chirp-tweety-plush-21-in/kid/softtoy001137). I chose this because it's a cute and cuddly plush toy that avoids her dislikes (nuts and flowers). Although it's not directly related to her likes (dark chocolate and orchids), it's a charming gift that could bring a smile to her face.

As alternatives, I also considered 

## Test 2 — Preference update

In [6]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='My wife loves dark chocolate and orchids, budget around LKR 5000',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 23:16:37.925 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 17:46:39 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 23:16:38.811 | INFO     | agents.router:classify:125 - Intent: preference_update (98%) — Providing recipient preferences and

Intent     : preference_update (98%)
Session    : s001
Metadata   : {'recipient': 'Wife', 'action': 'updated the profile', 'is_new_profile': False}

Reply:
Got it! I've updated the profile for Wife — I now know they likes dark chocolate, orchids and budget around LKR 5,000. I'll use this to personalise every gift recommendation. Would you like me to find something for them now?


## Test 3 — Search again (now personalised with saved profile)

In [7]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Find an anniversary gift for my wife',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 23:16:46.019 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 17:46:47 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 23:16:46.738 | INFO     | agents.router:classify:125 - Intent: search (98%) — Asking for gift recommendations for a specific

Intent     : search (98%)
Session    : s001
Metadata   : {'products_retrieved': 8, 'query_enriched': 'Find an anniversary gift for my wife | preferences: dark chocolate, orchids | avoid: nuts, flowers | budget LKR 5000', 'recipient': 'Wife', 'reflection_triggered': True, 'violations_found': True}

Reply:
I've found some wonderful anniversary gift options for your wife that cater to her love of dark chocolate and fit within your budget of LKR 5000. Since she dislikes nuts and flowers, I've carefully selected the following recommendations:

1. Java 08-piece Chocolates (LKR 6,200) - This exquisite chocolate set is a great way to show your love and appreciation. You can find it at https://www.kapruka.com/buyonline/java-08-piece-chocolates-with-/kid/chocolates001473.
2. Mother's Day Milk Chocolate Collection (LKR 4,800) - A thoughtful and delicious gift, this chocolate collection is sure to bring a smile to her face. You can purchase it at https://www.kapruka.com/buyonline/mother-s-day-milk

## Test 4 — Logistics check

In [8]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Can you deliver to Jaffna by this weekend?',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 23:17:06.392 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 17:47:07 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 23:17:07.216 | INFO     | agents.router:classify:125 - Intent: logistics_check (98%) — Asking about delivery feasibility to 

Intent     : logistics_check (98%)
Session    : s001
Metadata   : {'district': 'Jaffna', 'feasible': True}

Reply:
I'm happy to help you with delivery to Jaffna. Yes, we do deliver to Jaffna, and the estimated delivery time is 3 days. Please note that Jaffna falls under our extended delivery zone, which means it may take a little longer to reach you. There is a delivery surcharge of LKR 600 applicable to this region.

Since you're looking to get something delivered by this weekend, I would recommend placing your order as soon as possible to ensure timely arrival. As it's an extended zone, it's always a good idea to order well in advance, especially if you're planning for a special occasion with a fixed date.

Now that we've confirmed delivery, would you like some help finding a suitable gift? I'd be more than happy to assist you in choosing something perfect for your occasion. Just let me know what you're looking for, and I'll do my best to help!


## Test 5 — Logistics check (same-day zone)

In [9]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Do you deliver to Colombo today?',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 23:17:15.603 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 17:47:16 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 23:17:16.635 | INFO     | agents.router:classify:125 - Intent: logistics_check (99%) — Asking about same-day delivery to a s

Intent     : logistics_check (99%)
Session    : s001
Metadata   : {'district': 'Colombo', 'feasible': True}

Reply:
You're in luck. We do deliver to Colombo, and with our same-day delivery option, you can get your items delivered today if you order before 11 AM. Since it's a same-day delivery zone, you won't have to wait long to receive your order, and the best part is, there's no extra delivery surcharge. 

Now that we've confirmed delivery, would you like some help finding a perfect gift or finalizing your order? I'm here to assist you.


## Test 6 — Chitchat / greeting

In [10]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Hi! What can you help me with?',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 23:17:24.727 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 17:47:26 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 23:17:25.429 | INFO     | agents.router:classify:125 - Intent: chitchat (92%) — Greeting with no specific gift intent
2026-0

Intent     : chitchat (92%)
Session    : s001
Metadata   : {}

Reply:
Hello again, I see you have your lovely wife's profile saved with us. I'd be happy to help you find a perfect gift for her, considering her love for dark chocolate and orchids, and staying within your budget of LKR 5000 - would you like to browse some gift options or update her preferences?


## Test 7 — New recipient profile

In [11]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='My dad is turning 60, he loves cricket and books, budget LKR 8000',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 23:17:33.123 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 17:47:34 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 23:17:34.144 | INFO     | agents.router:classify:125 - Intent: preference_update (96%) — Sharing recipient's preferences, oc

Intent     : preference_update (96%)
Session    : s001
Metadata   : {'recipient': 'Dad', 'action': 'created a new profile', 'is_new_profile': True}

Reply:
Got it! I've created a new profile for Dad — I now know they likes cricket, books and budget around LKR 8,000. I'll use this to personalise every gift recommendation. Would you like me to find something for them now?


## Test 8 — Search for new recipient

In [12]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='What should I get dad for his birthday?',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 23:17:40.418 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 17:47:41 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 23:17:41.119 | INFO     | agents.router:classify:125 - Intent: search (98%) — Asking for gift recommendations with a previou

Intent     : search (98%)
Session    : s001
Metadata   : {'products_retrieved': 8, 'query_enriched': 'What should I get dad for his birthday? | preferences: cricket, books | budget LKR 8000', 'recipient': 'Dad', 'reflection_triggered': True, 'violations_found': False}

Reply:
I think I've found just the thing for your dad's birthday. My top recommendation is the Gift Box Sri Lanka (https://www.kapruka.com/buyonline/still-moments-with-bottle-and-/kid/giftset00591) priced at LKR 5,200. I chose this because it's a thoughtful and customized gift that shows you put extra effort into selecting something special for him. Although it doesn't directly relate to his interests in cricket or books, it's a versatile gift that can be appreciated by anyone.

Since none of the other products directly match your dad's interests in cricket, my next suggestions are a bit more niche. Product 5, "It'S Amma'S Birthday Tomorrow" (https://www.kapruka.com/buyonline/it-s-amma-s-birthday-tomorrow/kid/book002386)

---
## Part 4 — Reflection Loop Tests

**How it works:**
1. **Draft** — CatalogAgent generates initial recommendation
2. **Reflect** — LLM checks draft against recipient's dislikes/allergies
3. **Revise** — If violations found, LLM rewrites recommendation avoiding flagged products

Reflection only runs when the profile has `dislikes` or `notes` (allergies).

### Reflection Test 1 — Save profile with dislikes (triggers reflection)

In [13]:
# Save a profile with explicit dislikes — this makes the reflection loop run
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='My wife is allergic to nuts and hates flowers',
)
show(r)

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 23:17:55.544 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 17:47:56 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 23:17:56.570 | INFO     | agents.router:classify:125 - Intent: preference_update (96%) — Sharing recipient's allergies and d

Intent     : preference_update (96%)
Session    : s001
Metadata   : {'recipient': 'Wife', 'action': 'updated the profile', 'is_new_profile': False}

Reply:
Got it! I've updated the profile for Wife — I now know they dislikes nuts, flowers. I'll use this to personalise every gift recommendation. Would you like me to find something for them now?


### Reflection Test 2 — Search with dislike profile (reflection should trigger)

Watch the logs for:
- `Reflection: violations=True/False`

Check metadata for:
- `reflection_triggered: True` — reflection ran
- `violations_found: True/False` — whether draft was revised

In [14]:
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Find a gift for my wife',
)
show(r)

print('\n--- Reflection summary ---')
print('Reflection ran  :', r.metadata.get('reflection_triggered'))
print('Violations found:', r.metadata.get('violations_found'))

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 23:18:04.355 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 17:48:05 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 23:18:05.071 | INFO     | agents.router:classify:125 - Intent: search (99%) — Asking for gift recommendations for a specific

Intent     : search (99%)
Session    : s001
Metadata   : {'products_retrieved': 8, 'query_enriched': 'Find a gift for my wife | preferences: dark chocolate, orchids | avoid: nuts, flowers | budget LKR 5000', 'recipient': 'Wife', 'reflection_triggered': True, 'violations_found': True}

Reply:
I'd be happy to help you find a gift for your wife. Considering her love for dark chocolate and orchids, I've curated a list of thoughtful gift ideas that cater to her tastes.

One option is to consider a beautiful orchid-related gift, such as the 'Adaraniya Amma' Luxe Orchid Ribbon Mother'S Day Cake (LKR 7,800, https://www.kapruka.com/buyonline/adaraniya-amma-luxe-orchid-rib/kid/cake00ka002153). Although it's a bit above your budget, it's a unique and thoughtful gift that combines her love for orchids with a delicious cake.

If you're looking for a more affordable option, you might want to consider a dark chocolate gift. Unfortunately, none of the provided chocolate options specifically mention be

### Reflection Test 3 — No dislikes profile (reflection should NOT trigger)

In [15]:
# Dad profile has no dislikes → reflection_triggered should be False
r = orch.chat(
    user_id='u001',
    session_id='s001',
    message='Get a gift for my dad',
)
show(r)

print('\n--- Reflection summary ---')
print('Reflection ran  :', r.metadata.get('reflection_triggered'))  # Expected: False
print('Violations found:', r.metadata.get('violations_found'))      # Expected: False

Prompt 'gift-concierge-router-system-label:production' not found during refresh, evicting from cache.
2026-05-10 23:18:23.802 | DEBUG    | infastructure.observability:fetch_prompt:181 - fetch_prompt('gift-concierge-router-system') fell back to local: headers: {'date': 'Sun, 10 May 2026 17:48:25 GMT', 'content-type': 'application/json; charset=utf-8', 'content-length': '118', 'connection': 'keep-alive', 'x-robots-tag': 'noindex', 'x-content-type-options': 'nosniff', 'referrer-policy': 'strict-origin-when-cross-origin', 'document-policy': 'js-profiling', 'permissions-policy': 'autoplay=*, fullscreen=*, microphone=*', 'x-frame-options': 'SAMEORIGIN', 'vary': 'Origin, Accept-Encoding', 'etag': '"11yih926gdb3a"'}, status_code: 404, body: {'message': "Prompt not found: 'gift-concierge-router-system' with label 'production'", 'error': 'LangfuseNotFoundError'}
2026-05-10 23:18:24.550 | INFO     | agents.router:classify:125 - Intent: search (96%) — Asking for gift recommendations for a specific

Intent     : search (96%)
Session    : s001
Metadata   : {'products_retrieved': 8, 'query_enriched': 'Get a gift for my dad | preferences: cricket, books | budget LKR 8000', 'recipient': 'Dad', 'reflection_triggered': True, 'violations_found': False}

Reply:
I'd be happy to help you find a gift for your dad. Based on his interests in cricket and books, I've shortlisted a few options from our catalog.

My top recommendation is Product 3: https://www.kapruka.com/buyonline/mage-puththu-horu-aran-by-thil/kid/book001555, priced at LKR 1,500. This book seems like a great fit, given your dad's love for reading. Although it's not directly related to cricket, it's a thoughtful gift that caters to his interest in books. Plus, it's well within your budget of LKR 8,000.

If you'd like to consider other options, I can also suggest Product 5: https://www.kapruka.com/buyonline/mother-s-day-blush-romance-dip/kid/chocolates00ka00213, priced at LKR 5,450, or Product 7: https://www.kapruka.com/buyonline/

---
## View conversation history

In [16]:
from memory.st_store import SupabaseSTStore

st = SupabaseSTStore()
history = st.recent('u001', 's001', k=20)

for turn in history:
    role = turn.role.upper()
    print(f'[{role}] {turn.content[:120]}')
    print()

[USER] My wife loves dark chocolate and orchids, budget around LKR 5000

[ASSISTANT] Got it! I've updated the profile for Wife — I now know they likes dark chocolate, orchids and budget around LKR 5,000. I

[USER] Find an anniversary gift for my wife

[ASSISTANT] I've found some wonderful anniversary gift options for your wife that cater to her love of dark chocolate and fit within

[USER] Can you deliver to Jaffna by this weekend?

[ASSISTANT] I'm happy to help you with delivery to Jaffna. Yes, we do deliver to Jaffna, and the estimated delivery time is 3 days. 

[USER] Do you deliver to Colombo today?

[ASSISTANT] You're in luck. We do deliver to Colombo, and with our same-day delivery option, you can get your items delivered today 

[USER] Hi! What can you help me with?

[ASSISTANT] Hello again, I see you have your lovely wife's profile saved with us. I'd be happy to help you find a perfect gift for h

[USER] My dad is turning 60, he loves cricket and books, budget LKR 8000

[ASSIST

## View saved recipient profiles

In [17]:
from memory.profile_store import SupabaseProfileStore

ps = SupabaseProfileStore()
profiles = ps.list_profiles('u001')

for p in profiles:
    print(f'Name        : {p.name}')
    print(f'Relationship: {p.relationship}')
    print(f'Preferences : {p.preferences}')
    print(f'Dislikes    : {p.dislikes}')
    print(f'Budget LKR  : {p.budget_lkr}')
    print(f'Notes       : {p.notes}')
    print()

Name        : Dad
Relationship: dad
Preferences : ['cricket', 'books']
Dislikes    : []
Budget LKR  : 8000
Notes       : turning 60

Name        : Wife
Relationship: wife
Preferences : ['dark chocolate', 'orchids']
Dislikes    : ['nuts', 'flowers']
Budget LKR  : 5000
Notes       : None

